# Step 60 Fit Final K3 Fusion HMM

## What This Notebook Does

This is the first public Stage-6 notebook. It runs the final full-data K = 3 fusion HMM fit on the canonical Stage-4 retained dataset.

It preserves the existing final-fit workflow that:
- loads the canonical intermediate + nolags + minlen15 segment dataset from Stage 4,
- fits the full-data K = 3 HMM,
- performs seed screening and refit,
- writes the authoritative final/ model artifacts,
- and exports per-run gamma/ and viterbi/ outputs.

## When To Run It

Run this notebook after the canonical Stage-4 retained dataset is ready.

The public default remains:
- DATA_VARIANT = "intermediate"
- FEATURE_MODE = "nolags"
- MINLEN = 15
- K_FINAL = 3

## Inputs Expected

- the Stage-4 retained dataset root containing segments_manifest.tsv
- TensorFlow and osl_dynamics in the active Python environment

## Outputs Written

- root-level provenance and QC files under the chosen final-model output folder
- authoritative model outputs under final/
- per-run decoded outputs under gamma/ and viterbi/

## Environment Notes

This is the main GPU-sensitive Stage-6 fit step. CPU fallback may work, but it is usually much slower and less practical for the full final fit.

## Scientific Notes

- Stage 6 depends directly on the canonical Stage-4 dataset.
- Stage 5 contributes the scientific decision to carry K = 3 forward, but it is not a file dependency here.
- The dense runtime code now lives in a same-directory Python backend module so this notebook stays readable.

In [ ]:
# Step 0. Import Python modules and locate the active Stage-6 helper module

import sys
from pathlib import Path

STAGE6_DIR = Path.cwd()
candidate = Path.cwd() / "notebooks" / "6_hmm_final"
if not (STAGE6_DIR / "stage6_hmm_final_helpers.py").exists() and candidate.exists():
    STAGE6_DIR = candidate

if not (STAGE6_DIR / "stage6_hmm_final_helpers.py").exists():
    raise FileNotFoundError(
        f"Stage-6 helper not found: {STAGE6_DIR / 'stage6_hmm_final_helpers.py'}"
    )

if str(STAGE6_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE6_DIR.resolve()))
from stage6_hmm_final_helpers import ensure_configured_path, run_public_final_fit_step


In [ ]:
# Step 1. User-editable roots and final-fit settings

SEGMENTS_ROOT = Path("<SET_SEGMENTS_ROOT>")
FINAL_MODEL_ROOT = Path("<SET_FINAL_MODEL_ROOT>")

DATA_VARIANT = "intermediate"
FEATURE_MODE = "nolags"
MINLEN = 15
K_FINAL = 3

# Optional explicit manifest override.
MANIFEST_TSV = None

# Keep None to rely on TensorFlow memory-growth instead of a hard memory cap.
GPU_MEMORY_LIMIT_MB = None

# Keep the decoded outputs enabled unless you have a strong reason to change them.
SAVE_GAMMA = True
DO_VITERBI = True

SEGMENT_BRANCH_ROOT = SEGMENTS_ROOT / DATA_VARIANT
FINAL_MODEL_OUTPUT_ROOT = FINAL_MODEL_ROOT / f"PipelineE_final_K{K_FINAL:02d}_{DATA_VARIANT}_{FEATURE_MODE}_minlen{MINLEN}"

In [ ]:
# Step 2. Validate the configured inputs and print a short run summary

SEGMENTS_ROOT = ensure_configured_path(SEGMENTS_ROOT, "SEGMENTS_ROOT", must_exist=True)
FINAL_MODEL_ROOT = ensure_configured_path(FINAL_MODEL_ROOT, "FINAL_MODEL_ROOT")
if MANIFEST_TSV is not None:
    MANIFEST_TSV = ensure_configured_path(MANIFEST_TSV, "MANIFEST_TSV", must_exist=True)

print("Stage 6 / Step 60: final full-data K=3 fusion HMM fit")
print(f"  Stage-4 segment root:     {SEGMENT_BRANCH_ROOT}")
print(f"  Output directory:         {FINAL_MODEL_OUTPUT_ROOT}")
print(f"  Feature mode:             {FEATURE_MODE}")
print(f"  Minimum segment length:   {MINLEN} TR")
print(f"  Final chosen K:           {K_FINAL}")
print(f"  GPU memory cap (MB):      {GPU_MEMORY_LIMIT_MB}")
print(f"  Save gamma outputs:       {SAVE_GAMMA}")
print(f"  Save Viterbi outputs:     {DO_VITERBI}")
print("  Packages needed:          tensorflow, osl_dynamics, numpy, pandas")

In [ ]:
# Step 3. Run the final-fit backend

outputs = run_public_final_fit_step(
    segments_root=SEGMENTS_ROOT,
    final_model_root=FINAL_MODEL_ROOT,
    data_variant=DATA_VARIANT,
    feature_mode=FEATURE_MODE,
    minlen=MINLEN,
    k_final=K_FINAL,
    manifest_tsv=MANIFEST_TSV,
    gpu_memory_limit_mb=GPU_MEMORY_LIMIT_MB,
    save_gamma=SAVE_GAMMA,
    do_viterbi=DO_VITERBI,
)

outputs

## Notes

- This notebook keeps the public setup and scientific contract visible, but the dense runtime code now lives in stage6_final_fit_backend.py.
- The saved final-fit artifacts from this step are the inputs for the remaining public Stage-6 notebooks.